In [ ]:
from typing import Dict, TypedDict, List, Annotated, Sequence, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import ToolNode
import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
class AgentState(TypedDict):
    RegionName: Annotated[str, "The name of the district we are monitoring"]
    congestion_level: Annotated[Literal['None', 'Low', 'Medium', 'High'], "The real-time congestion level from the insights stream"]
    budget_remaining: Annotated[int, "The number of Location Retrieval API calls remaining for this hour"]
    net_accumulation: Annotated[int, "Net crowd size (people entering - people leaving) inside the zone geofence"]
    sentinel_pool: Annotated[List[str], "List of hashed device IDs currently acting as sentinels in the district"]
    verified_numbers: Annotated[List[str], "List of real phone numbers acquired from the vault that are reachable and in the danger zone"]
    density: Annotated[float, "Calculated people per square meter after Location Retrieval and DBSCAN"]
    time_to_critical: Annotated[float, "Minutes until the crowd reaches dangerous compressive asphyxia levels"]
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [3]:
@tool
def arm_chokepoints(zone_id: str) -> str:
    """
    Subscribes to zone geofences at chokepoints and calculates the net crowd accumulation.
    Call this to begin the escalation ladder when you decide an anomaly is worth investigating.
    """
    # Mocking the calculation: people entering - people leaving
    return f"Net accumulation for {zone_id} is +45 people/min"

@tool
def verify_and_filter(hashed_ids: List[str]) -> str:
    """
    Passes hashed IDs to the phone vault, drops offline devices (reachability check),
    and calls verify_v1 to confirm they are in the danger zone.
    Call this when accumulation is positive.
    """
    return "Verification complete. 50 actionable real numbers acquired."

@tool
def retrieve_locations_batch(batch_size: int) -> str:
    """
    Calls the Location Retrieval API for a small batch of phones (e.g. 5).
    Deterministically runs DBSCAN, calculates density, and time-to-critical.
    Call this after verification to locate the crowd.
    """
    return "Batch processed. Density: 4.2 people/m^2. Time to critical: 6 minutes."

In [ ]:
tools = [arm_chokepoints, verify_and_filter, retrieve_locations_batch]
model = ChatGoogleGenerativeAI(
    model="llama3-70b-8192", # Using a stable Llama 3 Groq model that usually has broad access
    temperature=0
).bind_tools(tools)

In [5]:
def agent_node(state: AgentState) -> dict:
    sys_prompt = SystemMessage(content="""
    You are an autonomous API budget manager for a smart city crowd safety system. 
    You will receive incoming webhook payloads containing real-time congestion data. 
    
    You MUST reason about the situation before spending API budget:
    1. CROSS-CHECK: If ALL districts read 'High', it is a network glitch. Output "Network glitch detected, suppressing alarm." and do NOT call tools.
    2. BASELINE CHECK: Compare the real-time congestion to the historical baseline for that specific time. If the high congestion is expected (e.g., normal egress after a match), keep budget in reserve.
    3. ESCALATE: If the congestion deviates significantly from the baseline (abnormal), escalate by calling the tools in this order: arm_chokepoints -> verify_and_filter -> retrieve_locations_batch.
    
    Before calling any tools, always write a short, 1-sentence reasoning trace explaining your decision.
    """)
    
    messages = [sys_prompt] + state.get("messages", [])
    response = model.invoke(messages)
    return {"messages": [response]}

In [6]:
def execute_tools(state: AgentState) -> dict:
    """
    Custom tool node that executes the requested tool and also explicitly updates
    the AgentState variables like time_to_critical and net_accumulation.
    """
    messages = state["messages"]
    last_message = messages[-1]
    
    state_updates = {}
    tool_messages = []
    
    for tool_call in last_message.tool_calls:
        if tool_call["name"] == "arm_chokepoints":
            res = arm_chokepoints.invoke(tool_call["args"])
            tool_messages.append(ToolMessage(content=res, tool_call_id=tool_call["id"]))
            state_updates["net_accumulation"] = 45  
            
        elif tool_call["name"] == "verify_and_filter":
            res = verify_and_filter.invoke(tool_call["args"])
            tool_messages.append(ToolMessage(content=res, tool_call_id=tool_call["id"]))
            state_updates["verified_numbers"] = ["+123", "+456"]  
            
        elif tool_call["name"] == "retrieve_locations_batch":
            res = retrieve_locations_batch.invoke(tool_call["args"])
            tool_messages.append(ToolMessage(content=res, tool_call_id=tool_call["id"]))
            state_updates["density"] = 4.2
            state_updates["time_to_critical"] = 6.0  
            
    return {"messages": tool_messages, **state_updates}

In [7]:
def should_continue(state: AgentState) -> Literal["tools", "evaluate_criticality"]:
    messages = state["messages"]
    last_message = messages[-1]
    
    if hasattr(last_message, 'tool_calls') and len(last_message.tool_calls) > 0:
        return "tools"
        
    return "evaluate_criticality"

In [8]:
def evaluate_criticality(state: AgentState) -> dict:
    """
    Deterministic rule engine that makes the final, auditable life-safety decision.
    AI must NOT make this decision.
    """
    time_to_critical = state.get("time_to_critical", None)
    
    if time_to_critical is not None and time_to_critical < 10.0:
        return {"messages": [SystemMessage(content=f"[DETERMINISTIC ALARM]: time_to_critical is {time_to_critical} mins (< 10). Sending SMS rerouting instructions.")]}
    else:
        return {"messages": [SystemMessage(content="[DETERMINISTIC NODE]: Situation is normal or unconfirmed. Stand down.")]}

In [9]:
from IPython.display import Image, display

graph = StateGraph(AgentState)

graph.add_node("agent", agent_node)
graph.add_node("tools", execute_tools) 
graph.add_node("evaluate_criticality", evaluate_criticality)

graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", should_continue)
graph.add_edge("tools", "agent")
graph.add_edge("evaluate_criticality", END)

app = graph.compile()

In [10]:
# Mocking a rich incoming webhook payload that includes baselines and other districts
webhook_payload = """
Time: 22:15 (Match ended at 22:00)
- Stadium District: Congestion is High (Historical Baseline at 22:15 is High)
- Marina District: Congestion is High (Historical Baseline at 22:15 is Low)
- Downtown District: Congestion is Low
- API Budget Remaining: 400 calls

Evaluate the situation and take action.
"""

initial_state = {
    "RegionName": "Multiple",
    "congestion_level": "High",
    "budget_remaining": 400,
    "messages": [HumanMessage(content=webhook_payload)]
}

for event in app.stream(initial_state, stream_mode="values"):
    if "messages" in event and event["messages"]:
        last_msg = event["messages"][-1]
        
        if hasattr(last_msg, 'content') and last_msg.content:
            print(f"-> [Agent Reasoning]: {last_msg.content}")
            
        if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
            for tool in last_msg.tool_calls:
                print(f"-> [Action]: Calling Tool -> {tool['name']}")

-> [Agent Reasoning]: 
Time: 22:15 (Match ended at 22:00)
- Stadium District: Congestion is High (Historical Baseline at 22:15 is High)
- Marina District: Congestion is High (Historical Baseline at 22:15 is Low)
- Downtown District: Congestion is Low
- API Budget Remaining: 400 calls

Evaluate the situation and take action.



BadRequestError: Error code: 400 - {'error': {'message': 'The model `llama3-70b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}